# Test Model Notebook

This notebook is used to visually compare model output results with other models / classic algorithms / ground truth

In [ ]:
import sys
sys.path.append("..")

import torch
from torch.utils.data import DataLoader

from PIL import Image

from model.dataset import SuperResDataset
from model.lit_upscaler import LitSuperResNet
from model.image_utils import ycbcr_tensor_to_pil, pil_to_lpips_tensor

import matplotlib.pyplot as plt

from skimage.io import imread
import os

import lpips

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# .env setup

This notebook uses .env file to get data paths from. This file is specific to the local machine and should be created manually.

**Make sure .env file is presented in the project root and following entries are added:**
- **TEST_DATA_PATH** - path to folder containing high-res images to test model on
- **_3RDPARTY_PATH** *(optional)* - path to folder containing results of upscaling same pictures by a 3rd party model
- **_3RDPARTY_MODEL** *(optional)* - name of a 3rd party model used in _3RDPARTY_PATH

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
TEST_PATH = os.getenv("TEST_DATA_PATH")
_3RDPARTY_PATH = os.getenv("_3RDPARTY_PATH")
_3RDPARTY_MODEL = os.getenv("_3RDPARTY_MODEL")

print(f"Test data path: {TEST_PATH}")

HIRES_PATCH_SIZE = 128

Here we define local model(s) to test. Models are loaded from checkpoints obtained by running `train.ipynb` or from the pre-trained model checkpoint provided with the project.

- `MODEL_A` path is required as the main model to test
- `MODEL_B` path is optional and could be used for comparing two local models side-by-side.

In [ ]:
MODEL_A = '../checkpoints/upscaler.ckpt'
MODEL_B = '../model/checkpoints/v14_tiny_loss/upscaler-epoch=630.ckpt'

upscaler_test = LitSuperResNet.load_from_checkpoint(MODEL_A).to(DEVICE)
upscaler_test.eval()

if MODEL_B is not None:
    upscaler_test_b = LitSuperResNet.load_from_checkpoint(MODEL_B).to(DEVICE)
    upscaler_test_b.eval()
else:
    upscaler_test_b = None

print("Test models are loaded")

# Testing

Here we compare different models, algorithms and ground truth patches side-by-side. 

**These patches are compared:**
- **Ground Truth** (real image patches from the test dataset)
- **Model A** + Model B outputs (if available)
- **3rd party model** results (if available)
- **Classic upscale algorithm** (Lanczos)

In addition to visual comparison, [**LPIPS**](https://richzhang.github.io/PerceptualSimilarity/) (learned perceptual image patch similarity) score is used.

In [ ]:
import torch.nn.functional as F

TEST_SEED = 42

lpips_fn = lpips.LPIPS(net='alex').to(DEVICE).eval()

test_dataset = SuperResDataset(
    TEST_PATH, 
    patch_size=HIRES_PATCH_SIZE,
    seed=TEST_SEED,
    downscale=None,
)
test_loader = DataLoader(test_dataset,
                     batch_size=16, shuffle=False, num_workers=0)
batch = next(iter(test_loader))

if _3RDPARTY_PATH is not None:
    _3rdparty_dataset = SuperResDataset(
        _3RDPARTY_PATH, 
        patch_size=HIRES_PATCH_SIZE,
        seed=TEST_SEED,
        downscale=None,
    )
    _3rdparty_loader = DataLoader(_3rdparty_dataset,
                     batch_size=16, shuffle=False, num_workers=0)
    _3rdparty_batch = next(iter(_3rdparty_loader))

with torch.no_grad():
    x, y_true = upscaler_test.batch_preprocess(batch)
    y_pred = upscaler_test(x.to(DEVICE)).clone()
    y_pred_b = upscaler_test_b(x.to(DEVICE)).clone() if upscaler_test_b is not None else None

for i in range(len(y_true)):

    lowres = ycbcr_tensor_to_pil(x[i].cpu())
    baseline = lowres.resize(
        (x.shape[-1] * 2, x.shape[-2] * 2),
        resample=Image.LANCZOS
    )
    
    gt = ycbcr_tensor_to_pil(y_true[i])

    panels = [
        (gt, "Real high-res"),
        (baseline, "LANCZOS"),
        (ycbcr_tensor_to_pil(y_pred[i]), "Model prediction"),
    ]

    if y_pred_b is not None:
        panels.insert(3, (ycbcr_tensor_to_pil(y_pred_b[i]), "Model prediction B"))

    if _3rdparty_batch is not None:
        panels.append((ycbcr_tensor_to_pil(_3rdparty_batch[i].cpu()), _3RDPARTY_MODEL))

    n = len(panels)
    dpi = 100
    w, h = panels[0][0].size
    DISPLAY_UPSCALE = 16
    fig, axes = plt.subplots(1, n, figsize=(DISPLAY_UPSCALE * n * w / dpi, DISPLAY_UPSCALE * h / dpi), dpi=dpi)

    if n == 1:
        axes = [axes]

    for ax, (img, title) in zip(axes, panels):
        if img != gt:
            with torch.no_grad():
                d = lpips_fn(pil_to_lpips_tensor(img, device=DEVICE), pil_to_lpips_tensor(gt, device=DEVICE))
                score = d.mean().item()
                title += f"\n(LPIPS: {score:.4f})"

        img = img.resize((h * DISPLAY_UPSCALE, w * DISPLAY_UPSCALE), resample=Image.NEAREST)
        ax.imshow(img, interpolation="nearest")
        ax.set_title(title, fontsize=72)
        ax.axis("off")

    plt.show()
